# Phase 8 — Spark UI, Monitoring & Debugging Experiments Notebook

This is the **STARTER notebook** for Phase 8.

Run it **top-to-bottom** while keeping the notebook kernel alive so the live Spark UI remains available.

The setup, deterministic retail datasets, helper functions, experiment order, runtime-evidence templates, and applied-project structure are preserved from the SOLUTION notebook.

Worked experiment code and answer-revealing explanations have been removed so you can perform the Spark UI diagnosis yourself.

Use this workflow throughout:

```text
predict
    ↓
run one known action
    ↓
job
    ↓
stage
    ↓
tasks
    ↓
record ACTUAL runtime evidence
    ↓
connect to df.explain('formatted')
    ↓
root cause
    ↓
change ONE thing when justified
    ↓
rerun
    ↓
compare
    ↓
reconcile correctness
```

Core question:

> **Where is the time or failure occurring, and what physical behavior caused it?**

Important:

- Do not invent Spark UI observations.
- Record what Spark actually shows for job IDs, stage IDs, task durations, shuffle sizes, spill, failures, executor behavior, and AQE changes.
- This notebook does not perform the formal mastery gate, update `ROADMAP.md`, mark Phase 8 complete, or enter Phase 9.


<a id="toc"></a>
## Table of Contents

- [Setup and Practice Data](#setup-and-practice-data)
- [Experiment Protocol](#experiment-protocol)
- [Experiment 1 — Multiple Actions and Jobs](#experiment-1)
- [Experiment 2 — Stage Boundaries and Shuffle Evidence](#experiment-2)
- [Experiment 3 — Balanced Tasks vs. Skewed Stragglers](#experiment-3)
- [Experiment 4 — Too Few vs. Too Many Tasks](#experiment-4)
- [Experiment 5 — Sort-Merge vs. Broadcast Join](#experiment-5)
- [Experiment 6 — Spill and Memory Pressure](#experiment-6)
- [Experiment 7 — Failed Task Localization](#experiment-7)
- [Experiment 8 — Executor Utilization and Driver Misuse](#experiment-8)
- [Experiment 9 — AQE Runtime Behavior](#experiment-9)
- [Experiment 10 — Recomputed Lineage vs. Cache Reuse](#experiment-10)
- [Applied Phase 8 Project](#applied-project)
- [Cleanup](#cleanup)

---

<a id="setup-and-practice-data"></a>
# Setup and Practice Data

Main grains:

```text
fact_sales_df
= one row per sale_id

dim_store_df
= one row per store_id

dim_product_df
= one row per product_id

balanced_sales_df / skewed_sales_df
= one row per sale_id
```


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


spark = (
    SparkSession.builder
    .appName('phase_08_spark_ui_experiments')
    .master('local[4]')
    # Keep ordinary experiments static so stage/task behavior is easier to compare.
    .config('spark.sql.shuffle.partitions', '12')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

print(f'Live Spark UI: {spark.sparkContext.uiWebUrl}')
print('Keep this notebook kernel running while you inspect completed jobs/stages.')


In [ ]:
# Grain: one row per sale_id.
fact_sales_df = (
    spark.range(
        start=0,
        end=240000,
        step=1,
        numPartitions=12,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.date_add(
            F.lit('2026-01-01').cast('date'),
            (F.col('id') % 180).cast('int'),
        ).alias('order_date'),
        F.concat(
            F.lit('S'),
            F.lpad(
                ((F.col('id') % 40) + F.lit(1)).cast('string'),
                3,
                '0',
            ),
        ).alias('store_id'),
        F.concat(
            F.lit('P'),
            F.lpad(
                ((F.col('id') % 500) + F.lit(1)).cast('string'),
                4,
                '0',
            ),
        ).alias('product_id'),
        F.concat(
            F.lit('C'),
            F.lpad(
                ((F.col('id') % 10000) + F.lit(1)).cast('string'),
                5,
                '0',
            ),
        ).alias('customer_id'),
        F.when(
            (F.col('id') % 10) < 8,
            F.lit('COMPLETED'),
        )
        .when(
            (F.col('id') % 10) == 8,
            F.lit('CANCELLED'),
        )
        .otherwise(F.lit('RETURNED'))
        .alias('order_status'),
        ((F.col('id') % 5) + F.lit(1)).cast('int').alias('quantity'),
        (
            F.lit(5.00)
            + ((F.col('id') % 75) * F.lit(0.75))
        )
        .cast(DecimalType(12, 2))
        .alias('unit_price'),
    )
    .withColumn(
        'gross_sales',
        (F.col('quantity') * F.col('unit_price'))
        .cast(DecimalType(16, 2)),
    )
    .withColumn('year', F.year('order_date'))
    .withColumn('month', F.month('order_date'))
)


# Grain: one row per store_id.
dim_store_df = (
    spark.range(
        start=1,
        end=41,
        step=1,
        numPartitions=2,
    )
    .select(
        F.concat(
            F.lit('S'),
            F.lpad(F.col('id').cast('string'), 3, '0'),
        ).alias('store_id'),
        F.concat(
            F.lit('Store '),
            F.col('id').cast('string'),
        ).alias('store_name'),
        F.when(F.col('id') <= 20, F.lit('ON'))
        .otherwise(F.lit('BC'))
        .alias('province'),
        F.when((F.col('id') % 2) == 0, F.lit('URBAN'))
        .otherwise(F.lit('SUBURBAN'))
        .alias('store_format'),
    )
)


# Grain: one row per product_id.
dim_product_df = (
    spark.range(
        start=1,
        end=501,
        step=1,
        numPartitions=4,
    )
    .select(
        F.concat(
            F.lit('P'),
            F.lpad(F.col('id').cast('string'), 4, '0'),
        ).alias('product_id'),
        F.concat(
            F.lit('Product '),
            F.col('id').cast('string'),
        ).alias('product_name'),
        F.concat(
            F.lit('CATEGORY_'),
            (F.col('id') % 12).cast('string'),
        ).alias('category'),
    )
)


# Grain: one row per sale_id; key frequencies are intentionally balanced.
balanced_sales_df = (
    spark.range(
        start=0,
        end=240000,
        step=1,
        numPartitions=12,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.concat(
            F.lit('S'),
            F.lpad(
                ((F.col('id') % 40) + F.lit(1)).cast('string'),
                3,
                '0',
            ),
        ).alias('store_id'),
        F.lit(1).cast('long').alias('units'),
    )
)


# Grain: one row per sale_id; S001 deliberately owns about 80% of rows.
skewed_sales_df = (
    spark.range(
        start=0,
        end=240000,
        step=1,
        numPartitions=12,
    )
    .select(
        F.col('id').cast('long').alias('sale_id'),
        F.when(
            (F.col('id') % 100) < 80,
            F.lit('S001'),
        )
        .otherwise(
            F.concat(
                F.lit('S'),
                F.lpad(
                    ((F.col('id') % 39) + F.lit(2)).cast('string'),
                    3,
                    '0',
                ),
            )
        )
        .alias('store_id'),
        F.lit(1).cast('long').alias('units'),
    )
)


In [ ]:
def run_action(label, action):
    '''Run one labelled materializing action and return result + elapsed seconds.'''

    # Make the action easier to identify in the Jobs view.
    spark.sparkContext.setJobDescription(label)

    started_at = perf_counter()
    result = action()
    elapsed_seconds = perf_counter() - started_at

    print(f'{label}: {elapsed_seconds:.3f} seconds')
    print('Treat wall-clock time as supporting evidence, not proof by itself.')

    return result, elapsed_seconds


def show_key_frequency(df, key_column, label):
    '''Show a small aggregated key-frequency diagnostic.'''

    print(f'\n{label}')

    (
        df
        .groupBy(key_column)
        .agg(F.count('*').alias('row_count'))
        .orderBy(
            F.col('row_count').desc(),
            F.col(key_column).asc_nulls_last(),
        )
        .show(20, truncate=False)
    )


def reconcile_scalar(left_df, right_df, measure_column, label):
    '''Assert that one aggregate business measure is identical.'''

    left_value = (
        left_df
        .agg(F.sum(measure_column).alias('measure'))
        .first()['measure']
    )

    right_value = (
        right_df
        .agg(F.sum(measure_column).alias('measure'))
        .first()['measure']
    )

    print(f'{label}: {left_value} == {right_value}')
    assert left_value == right_value


def assert_unique_key(df, key_column):
    '''Assert uniqueness for a dimension/business key.'''

    duplicate_count = (
        df
        .groupBy(key_column)
        .count()
        .filter(F.col('count') > 1)
        .count()
    )

    assert duplicate_count == 0


In [ ]:
# Freeze core correctness before performance experiments.
assert_unique_key(dim_store_df, 'store_id')
assert_unique_key(dim_product_df, 'product_id')

fact_row_count = fact_sales_df.count()
distinct_sale_id_count = fact_sales_df.select('sale_id').distinct().count()

assert distinct_sale_id_count == fact_row_count

print('Core grain / key checks passed.')


[Back to Table of Contents](#toc)

---

<a id="experiment-protocol"></a>
# Experiment Protocol

For every experiment:

1. State the required grain and correctness invariant.
2. Predict the physical behavior.
3. Inspect `df.explain('formatted')`.
4. Run one labelled materializing action.
5. Locate the corresponding job/query.
6. Find the slow/failing stage.
7. Compare task durations and data sizes.
8. Record actual shuffle/spill/failure/executor evidence.
9. Connect runtime evidence to the physical operator.
10. Change **one thing** only when evidence justifies it.
11. Rerun the same action.
12. Reconcile correctness.

Use this runtime record:

```text
RUN LABEL:
MATERIALIZING ACTION:
PREDICTION:

JOB / QUERY
- job ID(s):
- query ID:
- elapsed time:

STAGE
- stage ID:
- role:
- task count:
- stage duration:
- input:
- shuffle write:
- shuffle read:
- memory spill:
- disk spill:
- failed/retried tasks:

TASKS
- typical duration:
- maximum duration:
- typical data size:
- maximum data size:
- stragglers?:

EXECUTORS
- useful parallelism?:
- repeated slow/failing executor?:
- notable GC/spill?:

PLAN
- relevant operators:
- Exchange(s):
- join strategy:
- AQE evidence:

ROOT-CAUSE HYPOTHESIS:
```

**Never fill a runtime field from expectation. Record what the UI actually shows.**


[Back to Table of Contents](#toc)

---

<a id="experiment-1"></a>
# Experiment 1 — Multiple Actions and Jobs

## Question

Which action triggered the runtime work, and does a second action over the same uncached lineage recompute upstream work?

### Before execution

Answer:

```text
Where should a shuffle occur?
Why?
Should Action 2 automatically reuse Action 1's computed result?
What UI evidence would prove recomputation?
```


In [ ]:
# TODO — Experiment 1
# Build the completed-store-sales DataFrame, inspect its formatted physical plan,
# then run the two required labelled materializing actions.
# Record the actual Jobs / Stages evidence before moving on.


In [ ]:
# TODO — Experiment 1
# Build the completed-store-sales DataFrame, inspect its formatted physical plan,
# then run the two required labelled materializing actions.
# Record the actual Jobs / Stages evidence before moving on.


In [ ]:
# TODO — Experiment 1
# Build the completed-store-sales DataFrame, inspect its formatted physical plan,
# then run the two required labelled materializing actions.
# Record the actual Jobs / Stages evidence before moving on.


### Record actual evidence

```text
ACTION 1
- job ID(s):
- stage count:
- slowest stage:
- shuffle write/read:

ACTION 2
- job ID(s):
- stage count:
- slowest stage:
- shuffle write/read:

Does equivalent upstream aggregation work appear again?:
```

### Before execution

Answer:

```text
Where should a shuffle occur?
Why?
Should Action 2 automatically reuse Action 1's computed result?
What UI evidence would prove recomputation?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-2"></a>
# Experiment 2 — Stage Boundaries and Shuffle Evidence

## Question

Which stage is expensive, and what physical operator created that work?

### Before execution

Predict:

```text
Which operations may introduce stage boundaries?
Which physical operators should you look for?
What evidence would make one stage the main investigation target?
```


In [ ]:
# TODO — Experiment 2
# Build the completed-sales -> store join -> province aggregation pipeline.
# Inspect df.explain('formatted'), run the labelled action, then connect the
# slowest stage to the physical operator that created it.


In [ ]:
# TODO — Experiment 2
# Build the completed-sales -> store join -> province aggregation pipeline.
# Inspect df.explain('formatted'), run the labelled action, then connect the
# slowest stage to the physical operator that created it.


### Record actual evidence

```text
job/query:
source/input stage:
stage with shuffle write:
stage with shuffle read:
stage with final aggregation:
slowest stage:
task count in slowest stage:
typical task duration:
max task duration:
```

### Before execution

Predict:

```text
Which operations may introduce stage boundaries?
Which physical operators should you look for?
What evidence would make one stage the main investigation target?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-3"></a>
# Experiment 3 — Balanced Tasks vs. Skewed Stragglers

## Question

Does a verified hot key create measurably uneven task work?

### Before execution

Predict separately for balanced and skewed data:

```text
What does key-frequency evidence suggest?
Which Exchange key matters?
What task evidence would support or reject a skew diagnosis?
```


In [ ]:
# TODO — Experiment 3
# Run the balanced and skewed aggregation workloads separately.
# Verify key frequencies first, inspect each formatted plan, then record actual
# task-duration and shuffle-size distributions from the Spark UI.


In [ ]:
# TODO — Experiment 3
# Run the balanced and skewed aggregation workloads separately.
# Verify key frequencies first, inspect each formatted plan, then record actual
# task-duration and shuffle-size distributions from the Spark UI.


### Record actual comparison

| Evidence | Balanced | Skewed |
|---|---:|---:|
| Reduce-stage task count |  |  |
| Typical task duration |  |  |
| Max task duration |  |  |
| Typical shuffle read |  |  |
| Max shuffle read |  |  |
| Memory spill |  |  |
| Disk spill |  |  |

### Before execution

Predict separately for balanced and skewed data:

```text
What does key-frequency evidence suggest?
Which Exchange key matters?
What task evidence would support or reject a skew diagnosis?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-4"></a>
# Experiment 4 — Too Few vs. Too Many Tasks

## Question

Is parallelism limited by too few partitions, or is useful work fragmented into too many tiny tasks?

### Before execution

Predict:

```text
How can too few partitions limit parallelism?
How can too many tiny partitions create overhead?
Why must runtime task evidence decide whether either is a problem?
```


In [ ]:
# TODO — Experiment 4
# Create and run the deliberately under-partitioned and highly partitioned
# versions of the same workload. Compare task counts, data per task, durations,
# and useful parallelism.


In [ ]:
# TODO — Experiment 4
# Create and run the deliberately under-partitioned and highly partitioned
# versions of the same workload. Compare task counts, data per task, durations,
# and useful parallelism.


### Record actual comparison

```text
TWO PARTITIONS
- relevant stage tasks:
- typical data/task:
- typical task duration:
- executor capacity left idle?:

240 PARTITIONS
- relevant stage tasks:
- typical data/task:
- typical task duration:
- tasks extremely short?:
```

### Before execution

Predict:

```text
How can too few partitions limit parallelism?
How can too many tiny partitions create overhead?
Why must runtime task evidence decide whether either is a problem?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-5"></a>
# Experiment 5 — Sort-Merge vs. Broadcast Join

## Question

How does join strategy change the shuffle behavior visible in the UI?

### Correctness precondition

`dim_store_df` is unique on `store_id`, so the dimension join preserves fact grain before aggregation.

### Before execution

First validate join grain/key uniqueness. Then answer:

```text
What join strategy does the baseline plan actually show?
Which sides are redistributed?
What ONE join-strategy change would you test?
What runtime evidence should change if it works?
```


In [ ]:
# TODO — Experiment 5
# Validate the store dimension key, create the join baseline, then change ONE
# thing for the optimized join strategy. Compare shuffle evidence and reconcile
# the business result.


In [ ]:
# TODO — Experiment 5
# Validate the store dimension key, create the join baseline, then change ONE
# thing for the optimized join strategy. Compare shuffle evidence and reconcile
# the business result.


In [ ]:
# TODO — Experiment 5
# Validate the store dimension key, create the join baseline, then change ONE
# thing for the optimized join strategy. Compare shuffle evidence and reconcile
# the business result.


### Record actual comparison

| Evidence | Sort-merge baseline | Broadcast version |
|---|---:|---:|
| Join operator |  |  |
| Join-related Exchange(s) |  |  |
| Shuffle write |  |  |
| Shuffle read |  |  |
| Slowest relevant stage |  |  |
| Typical task duration |  |  |
| Max task duration |  |  |
| Spill |  |  |

### Before execution

First validate join grain/key uniqueness. Then answer:

```text
What join strategy does the baseline plan actually show?
Which sides are redistributed?
What ONE join-strategy change would you test?
What runtime evidence should change if it works?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-6"></a>
# Experiment 6 — Spill and Memory Pressure

## Question

If spill appears, which tasks spill and why?

### Before execution

Predict conditionally:

```text
If spill is isolated, what would you investigate?
If spill is widespread, what would you investigate?
If spill is zero, what can you legitimately conclude?
```


In [ ]:
# TODO — Experiment 6
# Build and run the aggregation candidate for spill investigation.
# Record actual memory/disk spill from the Spark UI. Zero spill is valid.


### Record actual evidence

```text
memory spill:
disk spill:
which tasks spill:
typical task duration:
max task duration:
typical shuffle read:
max shuffle read:
```

### Before execution

Predict conditionally:

```text
If spill is isolated, what would you investigate?
If spill is widespread, what would you investigate?
If spill is zero, what can you legitimately conclude?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-7"></a>
# Experiment 7 — Failed Task Localization

## Question

Can you localize a deterministic failure to the job, stage, task/partition, and expression?

### Before execution

Plan the failure investigation:

```text
How will you force evaluation of the failing expression?
How will you keep the application alive after the exception?
Which failure evidence will you preserve first?
```


In [ ]:
# TODO — Experiment 7
# Build the deterministic failure DataFrame and run an action that MUST evaluate
# the failing expression. Catch the exception, then localize job -> stage -> task.


In [ ]:
# TODO — Experiment 7
# Build the deterministic failure DataFrame and run an action that MUST evaluate
# the failing expression. Catch the exception, then localize job -> stage -> task.


### Record actual failure evidence

```text
job ID:
stage ID:
failing task / partition:
task attempt(s):
executor:
exception:
same logical task retried?:
```

### Before execution

Plan the failure investigation:

```text
How will you force evaluation of the failing expression?
How will you keep the application alive after the exception?
Which failure evidence will you preserve first?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-8"></a>
# Experiment 8 — Executor Utilization and Driver Misuse

## Question

Is the bottleneck distributed executor work, limited runnable parallelism, or driver-side behavior?


In [ ]:
# TODO — Experiment 8
# Run the executor-utilization workload, then create the safe driver diagnostic.
# Compare executor-side work with deliberately small driver-side collection.


### Record executor evidence

```text
active/completed tasks:
useful parallelism observed?:
one repeatedly slow/failing executor?:
notable GC?:
notable spill?:
```

### Before execution

Answer:

```text
What would useful executor utilization look like?
When can idle executors be caused by limited runnable tasks?
Why is collecting a small aggregate different from collecting the raw fact?
```


In [ ]:
# TODO — Experiment 8
# Run the executor-utilization workload, then create the safe driver diagnostic.
# Compare executor-side work with deliberately small driver-side collection.




### Before execution

Answer:

```text
What would useful executor utilization look like?
When can idle executors be caused by limited runnable tasks?
Why is collecting a small aggregate different from collecting the raw fact?
```


[Back to Table of Contents](#toc)

---

<a id="experiment-9"></a>
# Experiment 9 — AQE Runtime Behavior

## Question

Did AQE actually modify runtime execution?

### Before execution

Predict without assuming:

```text
What would prove AQE coalesced partitions?
What plan evidence would prove adaptation?
What should you conclude if AQE is enabled but runtime is unchanged?
```


In [ ]:
# TODO — Experiment 9
# Run the same aggregation with AQE disabled and enabled. Inspect the
# post-materialization plan and record whether AQE actually changed execution.


In [ ]:
# TODO — Experiment 9
# Run the same aggregation with AQE disabled and enabled. Inspect the
# post-materialization plan and record whether AQE actually changed execution.


### Record actual comparison

| Evidence | AQE off | AQE on |
|---|---:|---:|
| Configured shuffle target | 96 | 96 |
| Actual relevant task count |  |  |
| Adaptive/final plan evidence |  |  |
| Coalescing evidence |  |  |
| Stage runtime |  |  |

### Before execution

Predict without assuming:

```text
What would prove AQE coalesced partitions?
What plan evidence would prove adaptation?
What should you conclude if AQE is enabled but runtime is unchanged?
```


In [ ]:
# TODO — Experiment 9
# Run the same aggregation with AQE disabled and enabled. Inspect the
# post-materialization plan and record whether AQE actually changed execution.


[Back to Table of Contents](#toc)

---

<a id="experiment-10"></a>
# Experiment 10 — Recomputed Lineage vs. Cache Reuse

## Question

Does repeated runtime evidence justify persistence?

### Before execution

Answer:

```text
What proves uncached recomputation?
What work does cache materialization add?
What should change on reuse if caching is worthwhile?
```


In [ ]:
# TODO — Experiment 10
# Run two uncached actions, then persist the reusable intermediate. Distinguish
# cache materialization from cache reuse, inspect Storage, and unpersist.


In [ ]:
# TODO — Experiment 10
# Run two uncached actions, then persist the reusable intermediate. Distinguish
# cache materialization from cache reuse, inspect Storage, and unpersist.


### Record actual evidence

```text
UNCACHED ACTION 1
- expensive upstream stages:

UNCACHED ACTION 2
- repeated upstream stages?:

CACHE MATERIALIZATION
- materialization cost:
- Storage view cached partitions:

CACHE REUSE
- expensive upstream join/aggregation repeated?:
- new stage structure:
```

### Before execution

Answer:

```text
What proves uncached recomputation?
What work does cache materialization add?
What should change on reuse if caching is worthwhile?
```


In [ ]:
# TODO — Experiment 10
# Run two uncached actions, then persist the reusable intermediate. Distinguish
# cache materialization from cache reuse, inspect Storage, and unpersist.


[Back to Table of Contents](#toc)

---

<a id="applied-project"></a>
# Applied Phase 8 Project

## Scenario

Diagnose and improve:

```text
partitioned Parquet fact_sales
        ↓
Q1 2026 + COMPLETED filter
        ↓
store dimension join
        ↓
product dimension join
        ↓
province x category aggregation
        ↓
Parquet write
```

Business requirement:

```text
completed Q1 2026 gross sales by province and category
```

Required output grain:

```text
one row per province x category
```

The baseline deliberately forces a sort-merge join to the tiny store dimension.

Your job is not to optimize everything.

Your job is to use runtime evidence to determine the best **first** change.


In [ ]:
# TODO — Applied Phase 8 Project
# Implement the required step from the surrounding Markdown.
# Preserve the stated grain, one-change-at-a-time diagnosis, actual UI evidence,
# and before/after correctness reconciliation.


In [ ]:
# TODO — Applied Phase 8 Project
# Implement the required step from the surrounding Markdown.
# Preserve the stated grain, one-change-at-a-time diagnosis, actual UI evidence,
# and before/after correctness reconciliation.


In [ ]:
# TODO — Applied Phase 8 Project
# Implement the required step from the surrounding Markdown.
# Preserve the stated grain, one-change-at-a-time diagnosis, actual UI evidence,
# and before/after correctness reconciliation.


## Baseline runtime record

Fill from the actual Spark UI:

```text
ACTION
- labelled write:

JOB / QUERY
- job ID(s):
- query ID:
- wall-clock runtime:

SLOWEST STAGE
- stage ID:
- role:
- task count:
- stage duration:
- input:
- shuffle write:
- shuffle read:
- memory spill:
- disk spill:
- failed/retried tasks:

TASKS
- typical duration:
- max duration:
- typical data size:
- max data size:
- stragglers?:

EXECUTORS
- parallelism:
- repeated slow executor?:

PLAN
- PartitionFilters:
- PushedFilters:
- join 1:
- join 2:
- Exchange(s):
- aggregation:
```

## Before execution

Use this sequence:

```text
1. Freeze schema, grain, join keys, and business measure.
2. Inspect the baseline plan.
3. Run the labelled baseline write.
4. Find the relevant job/query and dominant stage.
5. Inspect tasks, shuffle, spill, executors, and scan evidence.
6. State ONE root-cause hypothesis.
7. Choose ONE first change supported by evidence.
8. Rerun the same write.
9. Reconcile correctness.
```

Complete before optimized code:

```text
PRIMARY BOTTLENECK:
EVIDENCE:
FIRST CHANGE:
WHY THIS CHANGE FIRST:
WHAT I AM DELIBERATELY NOT CHANGING YET:
```


In [ ]:
# TODO — Applied Phase 8 Project
# Implement the required step from the surrounding Markdown.
# Preserve the stated grain, one-change-at-a-time diagnosis, actual UI evidence,
# and before/after correctness reconciliation.


In [ ]:
# TODO — Applied Phase 8 Project
# Implement the required step from the surrounding Markdown.
# Preserve the stated grain, one-change-at-a-time diagnosis, actual UI evidence,
# and before/after correctness reconciliation.


In [ ]:
# TODO — Applied Phase 8 Project
# Implement the required step from the surrounding Markdown.
# Preserve the stated grain, one-change-at-a-time diagnosis, actual UI evidence,
# and before/after correctness reconciliation.


## Applied before/after evidence

Fill from the actual UI:

| Evidence | Baseline | Optimized |
|---|---:|---:|
| Join strategy for store dimension |  |  |
| Join-related Exchange(s) |  |  |
| Relevant stage runtime |  |  |
| Task count |  |  |
| Typical task duration |  |  |
| Maximum task duration |  |  |
| Shuffle write |  |  |
| Shuffle read |  |  |
| Memory spill |  |  |
| Disk spill |  |  |
| Failed/retried tasks |  |  |
| Wall-clock runtime |  |  |

## Before execution

Use this sequence:

```text
1. Freeze schema, grain, join keys, and business measure.
2. Inspect the baseline plan.
3. Run the labelled baseline write.
4. Find the relevant job/query and dominant stage.
5. Inspect tasks, shuffle, spill, executors, and scan evidence.
6. State ONE root-cause hypothesis.
7. Choose ONE first change supported by evidence.
8. Rerun the same write.
9. Reconcile correctness.
```

Complete before optimized code:

```text
PRIMARY BOTTLENECK:
EVIDENCE:
FIRST CHANGE:
WHY THIS CHANGE FIRST:
WHAT I AM DELIBERATELY NOT CHANGING YET:
```


In [ ]:
# TODO — Applied Phase 8 Project
# Implement the required step from the surrounding Markdown.
# Preserve the stated grain, one-change-at-a-time diagnosis, actual UI evidence,
# and before/after correctness reconciliation.


[Back to Table of Contents](#toc)

---

<a id="cleanup"></a>
# Cleanup

Run this only after you are finished inspecting the live Spark UI.

Stopping Spark removes the live application UI.


In [ ]:
# Remove temporary Parquet data created only for the applied experiment.
phase_08_temp_dir.cleanup()

# Stop Spark only when all UI inspection is finished.
spark.stop()


## Final Phase 8 diagnostic model

```text
symptom
→ action
→ job
→ stage
→ tasks
→ actual runtime evidence
→ physical plan
→ root cause
→ change ONE thing
→ rerun
→ compare
→ validate correctness
```

Core mastery requirement:

> **Explain why a Spark job is slow rather than merely observe that it is slow.**

[Back to Table of Contents](#toc)
